# ДЗ №4. Задание 1
### Выполнил: Лёвин В.А.
### Группа: ИУ6-54Б

Анализ MovieLens.


In [1]:
surname = "Лёвин" #Ваша фамилия

alp = 'абвгдеёжзийклмнопрстуфхцчшщъыьэюя'
w = [1, 42, 21, 21, 34,  6, 44, 26, 18, 44, 38, 26, 14, 43,  4, 49, 45,
        7, 42, 29,  4,  9, 36, 34, 31, 29,  5, 30,  4, 19, 28, 25, 33]

d = dict(zip(alp, w))
variant =  sum([d[el] for el in surname.lower()]) % 40 + 1

print("Задача № 1: ", variant % 3 + 1)
print("Задача № 2: ", variant % 2 + 1 )

Задача № 1:  3
Задача № 2:  1


## Подготовка

Вариант 3. Thriller, Sci-Fi, Adventure. У фильма с несколькими жанрами будет одна строка на каждый жанр.

In [2]:
from pyspark import SparkConf
from pyspark.sql import SparkSession, Window, functions as F

conf = SparkConf().setMaster("local[2]").setAppName("HW4-MovieLens")
conf.set("spark.sql.shuffle.partitions", "8")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
genres = ["Thriller", "Sci-Fi", "Adventure"]


def load_movies(folder):
    movies = spark.read.option("escape", '"').csv(f"{folder}/movies.csv", header=True)
    ratings = spark.read.csv(f"{folder}/ratings.csv", header=True)
    movie_genres = movies.select(
        F.col("movieId").cast("int").alias("movieId"), "title",
        F.explode(F.split("genres", r"\|")).alias("genre")
    )
    scores = ratings.groupBy(F.col("movieId").cast("int").alias("movieId")).agg(
        F.count("*").alias("rating_count"),
        F.avg(F.col("rating").cast("double")).alias("mean_rating")
    )
    stats = movie_genres.filter(F.col("genre").isin(genres)).join(scores, "movieId")
    return movie_genres, stats.cache()


def show_genres(movies):
    movies.groupBy("genre").count().orderBy(F.desc("count"), "genre").show(truncate=False)


def show_top(movies, order, *columns):
    window = Window.partitionBy("genre").orderBy(order, "movieId")
    top = movies.withColumn("rank", F.row_number().over(window)).filter("rank <= 10")
    top.orderBy("genre", "rank").select(
        "genre", "rank", "movieId", "title", *columns
    ).show(30)

26/11/19 09:15:32 WARN Utils: Your hostname, MacBook-Air-mac-4.local resolves to a loopback address: 127.0.0.1; using 172.16.216.45 instead (on interface en0)
26/11/19 09:15:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 09:15:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Малый набор

При равенстве значений первым идёт фильм с меньшим `movieId`.

In [3]:
movies, stats = load_movies("../ml-latest-small")

### 1. Выведите данные, сопоставляющие жанры и количество фильмов

In [4]:
show_genres(movies)

+------------------+-----+
|genre             |count|
+------------------+-----+
|Drama             |4361 |
|Comedy            |3756 |
|Thriller          |1894 |
|Action            |1828 |
|Romance           |1596 |
|Adventure         |1263 |
|Crime             |1199 |
|Sci-Fi            |980  |
|Horror            |978  |
|Fantasy           |779  |
|Children          |664  |
|Animation         |611  |
|Mystery           |573  |
|Documentary       |440  |
|War               |382  |
|Musical           |334  |
|Western           |167  |
|IMAX              |158  |
|Film-Noir         |87   |
|(no genres listed)|34   |
+------------------+-----+



### 2. Выведите первые 10 фильмов с наибольшим количеством рейтингов для каждого жанра в соответствии с вариантом

In [5]:
show_top(stats, F.desc("rating_count"), "rating_count")

+---------+----+-------+--------------------+------------+
|    genre|rank|movieId|               title|rating_count|
+---------+----+-------+--------------------+------------+
|Adventure|   1|    260|Star Wars: Episod...|         251|
|Adventure|   2|    480|Jurassic Park (1993)|         238|
|Adventure|   3|      1|    Toy Story (1995)|         215|
|Adventure|   4|   1196|Star Wars: Episod...|         211|
|Adventure|   5|    780|Independence Day ...|         202|
|Adventure|   6|    150|    Apollo 13 (1995)|         201|
|Adventure|   7|   1198|Raiders of the Lo...|         200|
|Adventure|   8|   4993|Lord of the Rings...|         198|
|Adventure|   9|   1210|Star Wars: Episod...|         196|
|Adventure|  10|   5952|Lord of the Rings...|         188|
|   Sci-Fi|   1|   2571|  Matrix, The (1999)|         278|
|   Sci-Fi|   2|    260|Star Wars: Episod...|         251|
|   Sci-Fi|   3|    480|Jurassic Park (1993)|         238|
|   Sci-Fi|   4|    589|Terminator 2: Jud...|         22

### 3. Выведите первые 10 фильмов с наименьшим количеством рейтингов (но больше 10) для каждого жанра в соответствии с вариантом

In [6]:
rated = stats.filter(F.col("rating_count") > 10)
show_top(rated, F.asc("rating_count"), "rating_count")

+---------+----+-------+--------------------+------------+
|    genre|rank|movieId|               title|rating_count|
+---------+----+-------+--------------------+------------+
|Adventure|   1|    599|Wild Bunch, The (...|          11|
|Adventure|   2|    631|All Dogs Go to He...|          11|
|Adventure|   3|    709|Oliver & Company ...|          11|
|Adventure|   4|   1216|Big Blue, The (Gr...|          11|
|Adventure|   5|   1615|    Edge, The (1997)|          11|
|Adventure|   6|   2135|Doctor Dolittle (...|          11|
|Adventure|   7|   2153|Avengers, The (1998)|          11|
|Adventure|   8|   2414|Young Sherlock Ho...|          11|
|Adventure|   9|   2822| Medicine Man (1992)|          11|
|Adventure|  10|   3259| Far and Away (1992)|          11|
|   Sci-Fi|   1|   2346|Stepford Wives, T...|          11|
|   Sci-Fi|   2|   2648| Frankenstein (1931)|          11|
|   Sci-Fi|   3|   2827|Astronaut's Wife,...|          11|
|   Sci-Fi|   4|   3503|Solaris (Solyaris...|          1

### 4. Выведите первые 10 фильмов с наибольшим средним рейтингом при количестве рейтингов больше 10 для каждого жанра в соответствии с вариантом

In [7]:
mean_rating = F.round("mean_rating", 3).alias("mean_rating")
show_top(rated, F.desc("mean_rating"), "rating_count", mean_rating)

+---------+----+-------+--------------------+------------+-----------+
|    genre|rank|movieId|               title|rating_count|mean_rating|
+---------+----+-------+--------------------+------------+-----------+
|Adventure|   1|   1204|Lawrence of Arabi...|          45|        4.3|
|Adventure|   2|   3508|Outlaw Josey Wale...|          18|       4.25|
|Adventure|   3|   1197|Princess Bride, T...|         142|      4.232|
|Adventure|   4|    260|Star Wars: Episod...|         251|      4.231|
|Adventure|   5|   3030|      Yojimbo (1961)|          13|      4.231|
|Adventure|   6|   1196|Star Wars: Episod...|         211|      4.216|
|Adventure|   7|   1198|Raiders of the Lo...|         200|      4.208|
|Adventure|   8|   2019|Seven Samurai (Sh...|          48|      4.188|
|Adventure|   9|    908|North by Northwes...|          57|      4.184|
|Adventure|  10|   1136|Monty Python and ...|         136|      4.162|
|   Sci-Fi|   1| 168252|        Logan (2017)|          25|       4.28|
|   Sc

### 5. Выведите первые 10 фильмов с наименьшим средним рейтингом при количестве рейтингов больше 10 для каждого жанра в соответствии с вариантом

In [8]:
show_top(rated, F.asc("mean_rating"), "rating_count", mean_rating)

+---------+----+-------+--------------------+------------+-----------+
|    genre|rank|movieId|               title|rating_count|mean_rating|
+---------+----+-------+--------------------+------------+-----------+
|Adventure|   1|   2643|Superman IV: The ...|          16|      1.688|
|Adventure|   2|   2422|Karate Kid, Part ...|          14|       1.75|
|Adventure|   3|   3997|Dungeons & Dragon...|          12|      1.833|
|Adventure|   4|   2404|    Rambo III (1988)|          12|      1.917|
|Adventure|   5|   1499|     Anaconda (1997)|          27|      1.926|
|Adventure|   6|    546|Super Mario Bros....|          21|        2.0|
|Adventure|   7|   3440|Teenage Mutant Ni...|          14|      2.071|
|Adventure|   8|  34520|Dukes of Hazzard,...|          12|      2.083|
|Adventure|   9|   2720|Inspector Gadget ...|          21|      2.095|
|Adventure|  10|   2450|Howard the Duck (...|          16|      2.156|
|   Sci-Fi|   1|   3593|Battlefield Earth...|          19|      1.658|
|   Sc

## Полный набор

In [9]:
stats.unpersist()
movies, stats = load_movies("../ml-latest")

### 1. Выведите данные, сопоставляющие жанры и количество фильмов

In [10]:
show_genres(movies)

+------------------+-----+
|genre             |count|
+------------------+-----+
|Drama             |33681|
|Comedy            |22830|
|Thriller          |11675|
|Romance           |10172|
|Action            |9563 |
|Documentary       |9283 |
|Horror            |8570 |
|(no genres listed)|7060 |
|Crime             |6917 |
|Adventure         |5349 |
|Sci-Fi            |4850 |
|Animation         |4579 |
|Children          |4367 |
|Mystery           |3972 |
|Fantasy           |3821 |
|War               |2301 |
|Western           |1690 |
|Musical           |1059 |
|Film-Noir         |354  |
|IMAX              |195  |
+------------------+-----+



### 2. Выведите первые 10 фильмов с наибольшим количеством рейтингов для каждого жанра в соответствии с вариантом

In [11]:
show_top(stats, F.desc("rating_count"), "rating_count")

+---------+----+-------+--------------------+------------+
|    genre|rank|movieId|               title|rating_count|
+---------+----+-------+--------------------+------------+
|Adventure|   1|    260|Star Wars: Episod...|       97202|
|Adventure|   2|    480|Jurassic Park (1993)|       83026|
|Adventure|   3|   1196|Star Wars: Episod...|       80200|
|Adventure|   4|   4993|Lord of the Rings...|       79940|
|Adventure|   5|      1|    Toy Story (1995)|       76813|
|Adventure|   6|   1210|Star Wars: Episod...|       76773|
|Adventure|   7|   7153|Lord of the Rings...|       75512|
|Adventure|   8|   1198|Raiders of the Lo...|       75248|
|Adventure|   9|   5952|Lord of the Rings...|       73687|
|Adventure|  10|   1270|Back to the Futur...|       67777|
|   Sci-Fi|   1|   2571|  Matrix, The (1999)|      107056|
|   Sci-Fi|   2|    260|Star Wars: Episod...|       97202|
|   Sci-Fi|   3|    480|Jurassic Park (1993)|       83026|
|   Sci-Fi|   4|   1196|Star Wars: Episod...|       8020

### 3. Выведите первые 10 фильмов с наименьшим количеством рейтингов (но больше 10) для каждого жанра в соответствии с вариантом

In [12]:
rated = stats.filter(F.col("rating_count") > 10)
show_top(rated, F.asc("rating_count"), "rating_count")

+---------+----+-------+--------------------+------------+
|    genre|rank|movieId|               title|rating_count|
+---------+----+-------+--------------------+------------+
|Adventure|   1|   3647| Running Free (1999)|          11|
|Adventure|   2|  46457| Triple Cross (1966)|          11|
|Adventure|   3|  48407|Conqueror, The (1...|          11|
|Adventure|   4|  77896|         Gold (1974)|          11|
|Adventure|   5|  87051|King of Escape, T...|          11|
|Adventure|   6|  89025|Evil - In the Tim...|          11|
|Adventure|   7|  92348|Puss in Boots (Na...|          11|
|Adventure|   8|  99089|Poor Little Rich ...|          11|
|Adventure|   9| 101335|  Stolen Seas (2012)|          11|
|Adventure|  10| 101967|Battle in Outer S...|          11|
|   Sci-Fi|   1|  27656|Resurrection of t...|          11|
|   Sci-Fi|   2|  51014|Strange Case of D...|          11|
|   Sci-Fi|   3|  66440|Robinson in Space...|          11|
|   Sci-Fi|   4|  69670|Final Approach (1...|          1

### 4. Выведите первые 10 фильмов с наибольшим средним рейтингом при количестве рейтингов больше 10 для каждого жанра в соответствии с вариантом

In [13]:
mean_rating = F.round("mean_rating", 3).alias("mean_rating")
show_top(rated, F.desc("mean_rating"), "rating_count", mean_rating)

+---------+----+-------+--------------------+------------+-----------+
|    genre|rank|movieId|               title|rating_count|mean_rating|
+---------+----+-------+--------------------+------------+-----------+
|Adventure|   1| 163809|Over the Garden W...|        1430|      4.257|
|Adventure|   2| 286897|Spider-Man: Acros...|         528|      4.253|
|Adventure|   3|   2019|Seven Samurai (Sh...|       17120|      4.251|
|Adventure|   4| 256991|Adventure Time: E...|          12|       4.25|
|Adventure|   5|   5618|Spirited Away (Se...|       35375|      4.226|
|Adventure|   6| 195159|Spider-Man: Into ...|       10885|      4.192|
|Adventure|   7|    908|North by Northwes...|       21883|      4.187|
|Adventure|   8|   6016|City of God (Cida...|       27209|      4.184|
|Adventure|   9| 228099|      Firefly (2002)|         895|      4.174|
|Adventure|  10|   3030|      Yojimbo (1961)|        5208|      4.167|
|   Sci-Fi|   1| 286897|Spider-Man: Acros...|         528|      4.253|
|   Sc

### 5. Выведите первые 10 фильмов с наименьшим средним рейтингом при количестве рейтингов больше 10 для каждого жанра в соответствии с вариантом

In [14]:
show_top(rated, F.asc("mean_rating"), "rating_count", mean_rating)
_ = stats.unpersist()

+---------+----+-------+--------------------+------------+-----------+
|    genre|rank|movieId|               title|rating_count|mean_rating|
+---------+----+-------+--------------------+------------+-----------+
|Adventure|   1| 171479|Kidnapping, Cauca...|          17|      0.912|
|Adventure|   2| 176005|    Titanic 2 (2010)|          22|      1.045|
|Adventure|   3| 103869|      Bigfoot (2012)|          17|      1.059|
|Adventure|   4| 128175|Beethoven's Treas...|          15|      1.167|
|Adventure|   5|   1826|Barney's Great Ad...|         463|      1.234|
|Adventure|   6|  31698|Son of the Mask (...|         847|      1.241|
|Adventure|   7| 130398|Transmorphers (2007)|          22|       1.25|
|Adventure|   8| 150230|Cool Cat Saves th...|          31|       1.29|
|Adventure|   9| 130384| Mega Piranha (2010)|          28|      1.393|
|Adventure|  10| 135216|The Star Wars Hol...|         248|      1.399|
|   Sci-Fi|   1| 206190|6-Headed Shark At...|          27|      0.907|
|   Sc